# 04 — Data Storage: Parquet (Bab 2.3)Notebook ini mengimplementasikan penyimpanan data ke format **Apache Parquet** dengan **partisi berdasarkan Product_Category**.

## 4.1 Inisialisasi & Load Data

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import col, to_date

spark = SparkSession.builder \
    .appName("04_DataStorage") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.driver.memory", "1g") \
    .getOrCreate()

schema = StructType([
    StructField("Transaction_ID", IntegerType(), False),
    StructField("Date", StringType(), False),
    StructField("Customer_ID", StringType(), False),
    StructField("Gender", StringType(), False),
    StructField("Age", IntegerType(), False),
    StructField("Product_Category", StringType(), False),
    StructField("Quantity", IntegerType(), False),
    StructField("Price_per_Unit", IntegerType(), False),
    StructField("Total_Amount", IntegerType(), False)
])

df = spark.read.csv("/data/retail_sales_dataset.csv", header=True, schema=schema)
df = df.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))
print(f"✅ Loaded {df.count()} rows")

## 4.2 Simpan ke Parquet (Partitioned)

In [ ]:
PARQUET_PATH = "/output/retail_parquet"

# Tulis ke Parquet, partisi berdasarkan Product_Category
df.write \
    .mode("overwrite") \
    .partitionBy("Product_Category") \
    .parquet(PARQUET_PATH)

print(f"✅ Data disimpan ke: {PARQUET_PATH}")
print(f"   Partisi: Product_Category (Beauty, Clothing, Electronics)")

## 4.3 Verifikasi: Baca Kembali dari Parquet

In [ ]:
# Baca kembali
df_parquet = spark.read.parquet(PARQUET_PATH)

print(f"✅ Parquet dibaca kembali: {df_parquet.count()} rows")
print(f"   Kolom: {df_parquet.columns}")
df_parquet.printSchema()
df_parquet.show(5)

## 4.4 Perbandingan Performa CSV vs Parquet

In [ ]:
import time

# Benchmark: CSV read
t0 = time.time()
df_csv = spark.read.csv("/data/retail_sales_dataset.csv", header=True, schema=schema)
df_csv.count()
csv_time = time.time() - t0

# Benchmark: Parquet read
t0 = time.time()
df_pq = spark.read.parquet(PARQUET_PATH)
df_pq.count()
pq_time = time.time() - t0

print("=== PERBANDINGAN PERFORMA ===")
print(f"   CSV read time     : {csv_time:.3f}s")
print(f"   Parquet read time : {pq_time:.3f}s")
print(f"   Speedup           : {csv_time/pq_time:.1f}x lebih cepat" if pq_time > 0 else "   (terlalu cepat untuk dibandingkan)")

## 4.5 Demo: Partition PruningKeunggulan partisi — hanya baca data yang dibutuhkan.

In [ ]:
# Hanya baca kategori 'Electronics' (partition pruning)
t0 = time.time()
df_elec = spark.read.parquet(PARQUET_PATH).filter(col("Product_Category") == "Electronics")
count_elec = df_elec.count()
prune_time = time.time() - t0

print(f"✅ Partition Pruning: hanya baca Electronics")
print(f"   Rows  : {count_elec}")
print(f"   Time  : {prune_time:.3f}s")
print(f"   → Spark hanya membaca folder 'Product_Category=Electronics', skip sisanya")